# AEZ-26 Phase 2D — Event Verification, Seasonality, and Compound-Driver Characterization

**Input:** authoritative Phase 2C v3 outputs  
**Primary events:** 39 regional AEZ-26 flash-drought episodes

This notebook:

- does **not** change the Phase 2C event labels;
- characterizes precipitation and atmospheric-demand forcing during the rapid-intensification window;
- summarizes monthly/seasonal occurrence;
- audits prolonged events;
- compares sensitivity catalogues;
- saves all tables and figures automatically; and
- creates one ZIP package for upload/audit.

Primary driver typology:

- standardized precipitation accumulation `P_z <= -1`
- standardized ERA5-Land potential evaporation `PE_z >= +1`

The potential-evaporation variable is an evaporative-demand proxy and is **not** FAO-56 ET0.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
PHASE2C_DIR = PHASE2_DIR / 'Phase2C_v3'
PHASE2D_DIR = PHASE2_DIR / 'Phase2D'
FIG_DIR = PHASE2D_DIR / 'Figures'

PHASE2D_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PENTAD_FILE = PHASE2C_DIR / 'AEZ26_Pentad_Hydroclimate_Indicators_2000_2025.csv'
PRIMARY_FILE = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Primary_2000_2025.csv'
ROOT_FILE = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_RootMoist_2000_2025.csv'
WINDOW_FILE = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_Window5_2000_2025.csv'
INDIA_FILE = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_India25_Max18_2000_2025.csv'

for p in [PENTAD_FILE, PRIMARY_FILE, ROOT_FILE, WINDOW_FILE, INDIA_FILE]:
    print(p.name, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(f'Missing Phase 2C v3 input: {p}')

print('Phase 2D output folder:', PHASE2D_DIR)


In [ ]:
pentad = pd.read_csv(
    PENTAD_FILE,
    parse_dates=['pentad_start', 'pentad_end', 'pentad_mid_date']
).sort_values('pentad_start').reset_index(drop=True)

primary = pd.read_csv(
    PRIMARY_FILE,
    parse_dates=[
        'anchor_mid_date',
        'onset_start_date',
        'onset_mid_date',
        'threshold_crossing_mid_date',
        'event_end_date',
        'termination_mid_date'
    ]
)

root = pd.read_csv(ROOT_FILE)
window5 = pd.read_csv(WINDOW_FILE)
india = pd.read_csv(INDIA_FILE)

assert len(pentad) == 1898
assert len(primary) == 39
assert pentad['pentad_label'].is_unique
assert primary['event_id'].is_unique

print('Pentad table:', pentad.shape)
print('Primary events:', len(primary))
print('RootMoist sensitivity:', len(root))
print('Window-5 sensitivity:', len(window5))
print('India-style sensitivity:', len(india))
print('Input gate: PASS')


## 1. Helpers for seasonal reference windows

For each event, the intensification interval is compared with the same climatological start pentad and same duration in each available year.


In [ ]:
label_to_idx = {
    label: idx
    for idx, label in enumerate(pentad['pentad_label'])
}

year_poy_to_idx = {
    (int(row.year), int(row.pentad_of_year)): idx
    for idx, row in pentad.iterrows()
}

study_years = sorted(pentad['year'].unique())


def sequential_year_pentads(start_year, start_pentad, n_pentads):
    sequence = []
    year = int(start_year)
    p = int(start_pentad)

    for _ in range(int(n_pentads)):
        sequence.append((year, p))
        p += 1

        if p > 73:
            p = 1
            year += 1

    return sequence


def reference_windows(start_pentad, n_pentads):
    refs = []

    for reference_year in study_years:
        seq = sequential_year_pentads(
            reference_year,
            start_pentad,
            n_pentads
        )

        if all(key in year_poy_to_idx for key in seq):
            refs.append(
                [year_poy_to_idx[key] for key in seq]
            )

    return refs


def day_weighted_mean(df, column):
    return float(
        np.average(
            df[column].astype(float),
            weights=df['n_days'].astype(float)
        )
    )


def standardized_stats(value, reference_values):
    ref = np.asarray(reference_values, dtype=float)
    ref = ref[np.isfinite(ref)]

    if len(ref) < 2:
        return np.nan, np.nan, np.nan, np.nan, len(ref)

    mean = ref.mean()
    sd = ref.std(ddof=1)

    z = (value - mean) / sd if sd > 0 else np.nan

    less = np.sum(ref < value)
    equal = np.sum(ref == value)
    percentile = 100.0 * (
        less + (equal + 1.0) / 2.0
    ) / len(ref)

    return float(z), float(percentile), float(mean), float(sd), int(len(ref))


def onset_season(month):
    month = int(month)

    if month in [12, 1, 2]:
        return 'Winter_DJF'
    if month in [3, 4, 5]:
        return 'PreMonsoon_MAM'
    if month in [6, 7, 8, 9]:
        return 'Monsoon_JJAS'
    return 'PostMonsoon_ON'


## 2. Independent Phase 2C event verification

Before driver characterization, reconstruct the event chronology from the pentad table so a corrupted event catalogue cannot pass silently.


In [ ]:
event_logic_violations = {
    'anchor_not_above_40': 0,
    'crossing_above_20': 0,
    'crossing_not_first_le20': 0,
    'rate_below_5': 0,
    'duration_below_3': 0,
    'recovery_below_20': 0,
    'event_overlap': 0
}

event_intervals = []

for ev in primary.itertuples(index=False):

    ai = label_to_idx[ev.anchor_pentad]
    oi = label_to_idx[ev.onset_pentad]
    ci = label_to_idx[ev.threshold_crossing_pentad]
    ei = label_to_idx[ev.event_end_pentad]

    event_logic_violations['anchor_not_above_40'] += int(
        ev.anchor_percentile <= 40
    )

    event_logic_violations['crossing_above_20'] += int(
        ev.threshold_crossing_percentile > 20
    )

    if ci > ai + 1:
        event_logic_violations['crossing_not_first_le20'] += int(
            (
                pentad.loc[
                    ai + 1:ci - 1,
                    'SM_0_100_pct'
                ] <= 20
            ).any()
        )

    event_logic_violations['rate_below_5'] += int(
        ev.mean_intensification_rate_pp_per_pentad < 5
    )

    event_logic_violations['duration_below_3'] += int(
        ev.total_duration_pentads < 3
    )

    if isinstance(ev.recovery_pentad, str):
        ri = label_to_idx[ev.recovery_pentad]
        event_logic_violations['recovery_below_20'] += int(
            pentad.loc[ri, 'SM_0_100_pct'] < 20
        )

    event_intervals.append((oi, ei))

for first, second in zip(event_intervals[:-1], event_intervals[1:]):
    event_logic_violations['event_overlap'] += int(
        second[0] <= first[1]
    )

print('Event-logic audit:')
for key, value in event_logic_violations.items():
    print(key, ':', value)

assert all(value == 0 for value in event_logic_violations.values())

print('Phase 2C event verification: PASS')


## 3. Build event-level driver characterization table

The primary forcing window is the **rapid-intensification interval** only: onset → first <=20th-percentile crossing.


In [ ]:
driver_rows = []

for ev in primary.itertuples(index=False):

    onset_idx = label_to_idx[ev.onset_pentad]
    crossing_idx = label_to_idx[ev.threshold_crossing_pentad]

    development = pentad.loc[
        onset_idx:crossing_idx
    ].copy()

    n_dev_pentads = len(development)
    actual_days = int(development['n_days'].sum())
    start_poy = int(development.iloc[0]['pentad_of_year'])

    # Phase 2C consistency.
    assert n_dev_pentads == int(
        ev.intensification_duration_pentads
    )

    # Event-period values.
    actual = {
        'P_sum': float(
            development['precipitation_sum_mm'].sum()
        ),
        'PE_sum': float(
            development[
                'ERA5_potential_evaporation_sum_mm'
            ].sum()
        ),
        'VPD_mean': day_weighted_mean(
            development,
            'VPD_mean_kPa'
        ),
        'T_mean': day_weighted_mean(
            development,
            't2m_mean_C'
        ),
        'Rn_mean': day_weighted_mean(
            development,
            'net_radiation_mean_MJ_m2_day'
        ),
        'Wind_mean': day_weighted_mean(
            development,
            'wind_speed_mean_m_s'
        ),
        'RH_mean': day_weighted_mean(
            development,
            'relative_humidity_mean_pct'
        )
    }

    refs = reference_windows(
        start_poy,
        n_dev_pentads
    )

    reference_values = {
        key: []
        for key in actual
    }

    for indices in refs:
        ref = pentad.loc[indices].copy()

        reference_values['P_sum'].append(
            float(ref['precipitation_sum_mm'].sum())
        )

        reference_values['PE_sum'].append(
            float(
                ref[
                    'ERA5_potential_evaporation_sum_mm'
                ].sum()
            )
        )

        reference_values['VPD_mean'].append(
            day_weighted_mean(
                ref,
                'VPD_mean_kPa'
            )
        )

        reference_values['T_mean'].append(
            day_weighted_mean(
                ref,
                't2m_mean_C'
            )
        )

        reference_values['Rn_mean'].append(
            day_weighted_mean(
                ref,
                'net_radiation_mean_MJ_m2_day'
            )
        )

        reference_values['Wind_mean'].append(
            day_weighted_mean(
                ref,
                'wind_speed_mean_m_s'
            )
        )

        reference_values['RH_mean'].append(
            day_weighted_mean(
                ref,
                'relative_humidity_mean_pct'
            )
        )

    stats = {
        key: standardized_stats(
            actual[key],
            reference_values[key]
        )
        for key in actual
    }

    P_z, P_pct, P_ref_mean, P_ref_sd, n_ref = stats['P_sum']
    PE_z, PE_pct, PE_ref_mean, PE_ref_sd, _ = stats['PE_sum']
    VPD_z, VPD_pct, _, _, _ = stats['VPD_mean']
    T_z, T_pct, _, _, _ = stats['T_mean']
    Rn_z, Rn_pct, _, _, _ = stats['Rn_mean']
    Wind_z, Wind_pct, _, _, _ = stats['Wind_mean']
    RH_z, RH_pct, _, _, _ = stats['RH_mean']

    # Primary standardized-anomaly typology.
    if P_z <= -1 and PE_z >= 1:
        driver_class = 'compound_Pdeficit_highPE'
    elif P_z <= -1:
        driver_class = 'precipitation_deficit_only'
    elif PE_z >= 1:
        driver_class = 'high_PE_only'
    else:
        driver_class = 'neither_large_anomaly'

    # Non-parametric 20/80 sensitivity typology.
    if P_pct <= 20 and PE_pct >= 80:
        percentile_class = 'compound_Pdeficit_highPE'
    elif P_pct <= 20:
        percentile_class = 'precipitation_deficit_only'
    elif PE_pct >= 80:
        percentile_class = 'high_PE_only'
    else:
        percentile_class = 'neither_large_anomaly'

    driver_rows.append({
        'event_id': ev.event_id,
        'onset_pentad': ev.onset_pentad,
        'onset_mid_date': ev.onset_mid_date,
        'onset_year': int(ev.onset_year),
        'onset_month': int(ev.onset_month),
        'onset_season': onset_season(ev.onset_month),

        'threshold_crossing_pentad':
            ev.threshold_crossing_pentad,

        'intensification_duration_pentads':
            int(ev.intensification_duration_pentads),

        'intensification_actual_days':
            actual_days,

        'total_duration_pentads':
            int(ev.total_duration_pentads),

        'severity_40pct_deficit_pp_pentads':
            float(ev.severity_40pct_deficit_pp_pentads),

        'mean_intensification_rate_pp_per_pentad':
            float(
                ev.mean_intensification_rate_pp_per_pentad
            ),

        'prolonged_after_flash_flag':
            int(ev.prolonged_after_flash_flag),

        'seasonal_reference_window_count':
            int(n_ref),

        'intensification_precipitation_mm':
            actual['P_sum'],

        'intensification_PE_mm':
            actual['PE_sum'],

        'intensification_VPD_mean_kPa':
            actual['VPD_mean'],

        'intensification_t2m_mean_C':
            actual['T_mean'],

        'intensification_net_radiation_mean_MJ_m2_day':
            actual['Rn_mean'],

        'intensification_wind_mean_m_s':
            actual['Wind_mean'],

        'intensification_RH_mean_pct':
            actual['RH_mean'],

        'P_reference_mean_mm':
            P_ref_mean,

        'P_reference_sd_mm':
            P_ref_sd,

        'PE_reference_mean_mm':
            PE_ref_mean,

        'PE_reference_sd_mm':
            PE_ref_sd,

        'P_standardized_anomaly_z':
            P_z,

        'PE_standardized_anomaly_z':
            PE_z,

        'VPD_standardized_anomaly_z':
            VPD_z,

        'temperature_standardized_anomaly_z':
            T_z,

        'net_radiation_standardized_anomaly_z':
            Rn_z,

        'wind_standardized_anomaly_z':
            Wind_z,

        'RH_standardized_anomaly_z':
            RH_z,

        'P_empirical_percentile':
            P_pct,

        'PE_empirical_percentile':
            PE_pct,

        'VPD_empirical_percentile':
            VPD_pct,

        'temperature_empirical_percentile':
            T_pct,

        'net_radiation_empirical_percentile':
            Rn_pct,

        'wind_empirical_percentile':
            Wind_pct,

        'RH_empirical_percentile':
            RH_pct,

        'large_precipitation_deficit_flag':
            int(P_z <= -1),

        'large_PE_anomaly_flag':
            int(PE_z >= 1),

        'high_VPD_anomaly_flag':
            int(VPD_z >= 1),

        'high_temperature_anomaly_flag':
            int(T_z >= 1),

        'high_net_radiation_anomaly_flag':
            int(Rn_z >= 1),

        'high_wind_anomaly_flag':
            int(Wind_z >= 1),

        'low_RH_anomaly_flag':
            int(RH_z <= -1),

        'primary_driver_class':
            driver_class,

        'percentile20_80_driver_class_sensitivity':
            percentile_class,

        'driver_class_sensitivity_agreement_flag':
            int(driver_class == percentile_class)
    })


event_driver = pd.DataFrame(driver_rows)

assert len(event_driver) == len(primary) == 39
assert event_driver['event_id'].is_unique
assert event_driver[
    'seasonal_reference_window_count'
].min() >= 20

z_columns = [
    'P_standardized_anomaly_z',
    'PE_standardized_anomaly_z',
    'VPD_standardized_anomaly_z',
    'temperature_standardized_anomaly_z',
    'net_radiation_standardized_anomaly_z',
    'wind_standardized_anomaly_z',
    'RH_standardized_anomaly_z'
]

assert np.isfinite(
    event_driver[z_columns].to_numpy(dtype=float)
).all()

display(event_driver.head())


## 4. Monthly, seasonal, annual, and driver-class summaries


In [ ]:
month_names = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

monthly = (
    event_driver
    .groupby('onset_month')
    .size()
    .reindex(range(1, 13), fill_value=0)
    .rename('event_count')
    .reset_index()
)

monthly['month_name'] = monthly['onset_month'].map(month_names)
monthly['event_percent'] = (
    100.0 * monthly['event_count'] / len(event_driver)
)

season_order = [
    'Winter_DJF',
    'PreMonsoon_MAM',
    'Monsoon_JJAS',
    'PostMonsoon_ON'
]

seasonal = (
    event_driver
    .groupby('onset_season')
    .size()
    .reindex(season_order, fill_value=0)
    .rename('event_count')
    .reset_index()
)

seasonal['event_percent'] = (
    100.0 * seasonal['event_count'] / len(event_driver)
)

annual = (
    event_driver
    .groupby('onset_year')
    .size()
    .reindex(range(2000, 2026), fill_value=0)
    .rename('event_count')
    .reset_index()
)

driver_order = [
    'compound_Pdeficit_highPE',
    'precipitation_deficit_only',
    'high_PE_only',
    'neither_large_anomaly'
]

driver_summary = (
    event_driver
    .groupby('primary_driver_class')
    .size()
    .reindex(driver_order, fill_value=0)
    .rename('event_count')
    .reset_index()
)

driver_summary['event_percent'] = (
    100.0 * driver_summary['event_count'] / len(event_driver)
)

percentile_driver_summary = (
    event_driver
    .groupby(
        'percentile20_80_driver_class_sensitivity'
    )
    .size()
    .reindex(driver_order, fill_value=0)
    .rename('event_count')
    .reset_index()
)

percentile_driver_summary['event_percent'] = (
    100.0
    * percentile_driver_summary['event_count']
    / len(event_driver)
)

display(monthly)
display(seasonal)
display(driver_summary)


## 5. Sensitivity-catalogue onset agreement

Exact and near-onset matching is one-to-one: a sensitivity event can match only one primary event.


In [ ]:
def pentad_serial(label):
    year = int(str(label)[:4])
    p = int(str(label).split('_P')[1])
    return (year - 2000) * 73 + (p - 1)


def greedy_onset_match(primary_labels, other_labels, tolerance):
    primary_serial = sorted(
        (pentad_serial(label), label)
        for label in primary_labels
    )

    other_serial = sorted(
        (pentad_serial(label), label)
        for label in other_labels
    )

    used = set()
    matches = []

    for p_serial, p_label in primary_serial:

        candidates = []

        for idx, (o_serial, o_label) in enumerate(other_serial):

            if idx in used:
                continue

            distance = abs(p_serial - o_serial)

            if distance <= tolerance:
                candidates.append(
                    (distance, idx, o_label)
                )

        if candidates:
            distance, idx, o_label = min(candidates)
            used.add(idx)

            matches.append({
                'primary_onset': p_label,
                'sensitivity_onset': o_label,
                'pentad_distance': distance
            })

    return matches


sensitivity_specs = [
    ('RootMoist', root),
    ('Window5', window5),
    ('India25_Max18', india)
]

agreement_rows = []

for name, df in sensitivity_specs:

    exact = greedy_onset_match(
        primary['onset_pentad'],
        df['onset_pentad'],
        tolerance=0
    )

    within1 = greedy_onset_match(
        primary['onset_pentad'],
        df['onset_pentad'],
        tolerance=1
    )

    within2 = greedy_onset_match(
        primary['onset_pentad'],
        df['onset_pentad'],
        tolerance=2
    )

    agreement_rows.append({
        'sensitivity_catalogue': name,
        'primary_event_count': len(primary),
        'sensitivity_event_count': len(df),
        'exact_onset_matches': len(exact),
        'exact_match_percent_of_primary':
            100.0 * len(exact) / len(primary),
        'within_1_pentad_matches': len(within1),
        'within_1_pentad_percent_of_primary':
            100.0 * len(within1) / len(primary),
        'within_2_pentad_matches': len(within2),
        'within_2_pentad_percent_of_primary':
            100.0 * len(within2) / len(primary)
    })

sensitivity_agreement = pd.DataFrame(agreement_rows)

display(sensitivity_agreement)


## 6. Prolonged-after-flash diagnostics

These events remain in the primary inventory, but their driver class is based only on the rapid-intensification window.


In [ ]:
prolonged = (
    event_driver.loc[
        event_driver['prolonged_after_flash_flag'] == 1
    ]
    .copy()
    .sort_values('onset_mid_date')
)

display(prolonged)


## 7. Phase 2D QC


In [ ]:
qc_rows = []


def add_qc(check, value, status, note):
    qc_rows.append({
        'check': check,
        'value': value,
        'status': status,
        'note': note
    })


add_qc(
    'Primary events retained',
    len(event_driver),
    'PASS',
    'Must remain 39; Phase 2D cannot alter labels.'
)

add_qc(
    'Unique event IDs',
    event_driver['event_id'].nunique(),
    'PASS',
    'Must equal 39.'
)

add_qc(
    'Event logic violations',
    sum(event_logic_violations.values()),
    'PASS',
    'Must be zero.'
)

add_qc(
    'Minimum seasonal reference windows',
    int(
        event_driver[
            'seasonal_reference_window_count'
        ].min()
    ),
    'PASS',
    'At least 20 required.'
)

add_qc(
    'Non-finite standardized anomalies',
    int(
        (~np.isfinite(
            event_driver[z_columns].to_numpy(dtype=float)
        )).sum()
    ),
    'PASS',
    'Must be zero.'
)

add_qc(
    'Driver-classified events',
    int(driver_summary['event_count'].sum()),
    'PASS',
    'Must equal 39.'
)

add_qc(
    'Monthly summary event total',
    int(monthly['event_count'].sum()),
    'PASS',
    'Must equal 39.'
)

add_qc(
    'Seasonal summary event total',
    int(seasonal['event_count'].sum()),
    'PASS',
    'Must equal 39.'
)

add_qc(
    'Annual summary event total',
    int(annual['event_count'].sum()),
    'PASS',
    'Must equal 39.'
)

add_qc(
    'Prolonged-after-flash events',
    len(prolonged),
    'RESULT',
    'Retained; diagnose separately.'
)

add_qc(
    'Primary-vs-percentile driver-class agreement',
    int(
        event_driver[
            'driver_class_sensitivity_agreement_flag'
        ].sum()
    ),
    'RESULT',
    'Sensitivity diagnostic; no tuning.'
)

qc = pd.DataFrame(qc_rows)

display(qc)

assert len(event_driver) == 39
assert event_driver['event_id'].nunique() == 39
assert sum(event_logic_violations.values()) == 0
assert driver_summary['event_count'].sum() == 39
assert monthly['event_count'].sum() == 39
assert seasonal['event_count'].sum() == 39
assert annual['event_count'].sum() == 39


## 8. Save tables automatically


In [ ]:
EVENT_DRIVER_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Event_Driver_Characterization.csv'
MONTHLY_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Monthly_Onset_Summary.csv'
SEASONAL_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Seasonal_Onset_Summary.csv'
ANNUAL_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Annual_Event_Counts_2000_2025.csv'
DRIVER_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Driver_Class_Summary.csv'
PCT_DRIVER_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Driver_Class_Percentile_Sensitivity.csv'
SENS_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Sensitivity_Agreement.csv'
PROLONGED_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Prolonged_Event_Diagnostics.csv'
QC_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_QC.csv'
CONFIG_OUT = PHASE2D_DIR / 'AEZ26_Phase2D_Method_Config.txt'

event_driver.to_csv(EVENT_DRIVER_OUT, index=False, date_format='%Y-%m-%d')
monthly.to_csv(MONTHLY_OUT, index=False)
seasonal.to_csv(SEASONAL_OUT, index=False)
annual.to_csv(ANNUAL_OUT, index=False)
driver_summary.to_csv(DRIVER_OUT, index=False)
percentile_driver_summary.to_csv(PCT_DRIVER_OUT, index=False)
sensitivity_agreement.to_csv(SENS_OUT, index=False)
prolonged.to_csv(PROLONGED_OUT, index=False, date_format='%Y-%m-%d')
qc.to_csv(QC_OUT, index=False)

method_config = '''PHASE 2D METHOD CONFIG
Primary event catalogue: Phase 2C v3 (39 events)
Event labels changed in Phase 2D: NO
Driver window: onset pentad through first <=20th-percentile crossing, inclusive
Primary precipitation diagnostic: standardized seasonal-window precipitation accumulation anomaly
Primary evaporative-demand diagnostic: standardized seasonal-window ERA5-Land potential-evaporation accumulation anomaly
Primary large precipitation threshold: z <= -1
Primary large potential-evaporation threshold: z >= +1
Precipitation index terminology: standardized accumulation anomaly; NOT SPI
ERA5 potential evaporation terminology: potential evaporation / evaporative-demand proxy; NOT FAO-56 ET0
Non-parametric sensitivity: precipitation <=20th percentile and potential evaporation >=80th percentile
Secondary atmospheric flags: VPD/T/Rn/wind z >= +1; RH z <= -1
Season groups: DJF, MAM, JJAS, ON
Formal annual trend inference in Phase 2D: NO
'''

CONFIG_OUT.write_text(method_config, encoding='utf-8')

print('Saved Phase 2D tables:')
for path in [
    EVENT_DRIVER_OUT,
    MONTHLY_OUT,
    SEASONAL_OUT,
    ANNUAL_OUT,
    DRIVER_OUT,
    PCT_DRIVER_OUT,
    SENS_OUT,
    PROLONGED_OUT,
    QC_OUT,
    CONFIG_OUT
]:
    print(' -', path)


## 9. Save diagnostic figures automatically

Figures are descriptive. They do not alter event definitions or driver thresholds.


In [ ]:
# Figure 1 — monthly onset counts
plt.figure(figsize=(9, 5))
plt.bar(monthly['month_name'], monthly['event_count'])
plt.xlabel('Onset month')
plt.ylabel('Number of primary events')
plt.title('AEZ-26 Flash-Drought Onset by Month (2000–2025)')
plt.tight_layout()

MONTH_FIG = FIG_DIR / 'AEZ26_FD_Onset_Monthly_Counts.png'
plt.savefig(MONTH_FIG, dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 2 — driver quadrants
plt.figure(figsize=(7, 6))
plt.scatter(
    event_driver['P_standardized_anomaly_z'],
    event_driver['PE_standardized_anomaly_z']
)
plt.axvline(-1)
plt.axhline(1)
plt.xlabel('Standardized precipitation accumulation anomaly (z)')
plt.ylabel('Standardized ERA5-Land potential-evaporation anomaly (z)')
plt.title('AEZ-26 Flash-Drought Driver Space')
plt.tight_layout()

DRIVER_FIG = FIG_DIR / 'AEZ26_FD_Driver_Quadrants_Pz_vs_PEz.png'
plt.savefig(DRIVER_FIG, dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 3 — annual event counts
plt.figure(figsize=(11, 5))
plt.bar(annual['onset_year'], annual['event_count'])
plt.xlabel('Year')
plt.ylabel('Number of primary events')
plt.title('AEZ-26 Annual Flash-Drought Event Counts')
plt.tight_layout()

ANNUAL_FIG = FIG_DIR / 'AEZ26_FD_Annual_Event_Counts.png'
plt.savefig(ANNUAL_FIG, dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 4 — duration versus severity
plt.figure(figsize=(7, 6))
plt.scatter(
    event_driver['total_duration_pentads'],
    event_driver['severity_40pct_deficit_pp_pentads']
)
plt.xlabel('Total event duration (pentads)')
plt.ylabel('Severity (percentile-point pentads)')
plt.title('AEZ-26 Flash-Drought Duration and Severity')
plt.tight_layout()

SEVERITY_FIG = FIG_DIR / 'AEZ26_FD_Duration_vs_Severity.png'
plt.savefig(SEVERITY_FIG, dpi=200, bbox_inches='tight')
plt.show()


## 10. Create one ZIP package for audit

After this cell runs, upload only the ZIP file to ChatGPT for independent Phase 2D audit.


In [ ]:
ZIP_BASE = PHASE2_DIR / 'AEZ26_Phase2D_Output_Package'

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    'zip',
    root_dir=PHASE2_DIR,
    base_dir='Phase2D'
)

print('ZIP CREATED:')
print(zip_path)


In [ ]:
print('FINAL PHASE 2D CHECK')
print('Primary events:', len(event_driver))
print()
print('Season counts:')
print(seasonal.to_string(index=False))
print()
print('Primary driver classes:')
print(driver_summary.to_string(index=False))
print()
print('Sensitivity agreement:')
print(sensitivity_agreement.to_string(index=False))
print()
print('Prolonged-after-flash events:', len(prolonged))
print()
print('Driver-class agreement with 20/80 percentile sensitivity:',
      f"{event_driver['driver_class_sensitivity_agreement_flag'].mean()*100:.1f}%")
print()
print('Upload this ZIP for independent audit:')
print(zip_path)


## Next gate

Do not proceed to MODIS/GRACE/JRC response alignment until the generated Phase 2D ZIP has been independently audited.

After Phase 2D passes, the next stage is **Phase 2E — native-time response alignment** for MOD16, MOD13, GRACE/GRACE-FO, and JRC, followed by the groundwater-observation branch.
